In [1]:
!python -m pip install .. --quiet

In [2]:
import ee 

ee.Authenticate() 
ee.Initialize(project='epistem2')

# 1. Satellite imagery

In [3]:
# AOI definition

aoi = ee.FeatureCollection('projects/epistem2/assets/AOI_Sumatra').geometry()


In [4]:

import geemap
from luma_ge.data_acquisition import Reflectance_Data, final_Image

#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2020-01-01'
end = '2020-12-31'
from luma_ge.data_acquisition import Reflectance_Data, final_Image

optical_reflectance = Reflectance_Data()

composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2020-01-01'
end = '2020-12-31'

landsat_data, meta = optical_reflectance.get_optical_data(
    aoi, start, end, optical_data='L8_SR', 
    compute_detailed_stats=False  # skip expensive aggregations
)

stacked_landsat = composite.get_quality_mosaic(
    landsat_data, aoi, 
    calculate_coverage=False  # skip pixel counting
)


2026-08-07 11:24:49,655 - luma_ge.ee_config - INFO - Earth Engine initialized successfully
2026-08-07 11:24:49,656 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-08-07 11:24:49,657 - final_Image - INFO - final_Image creation initialized.
2026-08-07 11:24:49,659 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-08-07 11:24:49,661 - final_Image - INFO - final_Image creation initialized.
2026-08-07 11:24:49,663 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-08-07 11:24:49,664 - Reflectance_Data - INFO - Date range: 2020-01-01 to 2020-12-31
2026-08-07 11:24:49,664 - Reflectance_Data - INFO - Cloud cover threshold: 30%
2026-08-07 11:24:49,666 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-08-07 11:24:49,667 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-08-07 11:24:49,667 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=Tr

# 2. Classification scheme 

In [5]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "Epistem"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df.to_string(index=False))

 ID          Land Cover Class Color Palette
  1    Primary Dryland Forest       #006400
  2  Secondary Dryland Forest       #228B22
  3   Primary Mangrove Forest       #4169E1
  4 Secondary Mangrove Forest       #87CEEB
  5      Primary Swamp Forest       #2E8B57
  6    Secondary Swamp Forest       #8FBC8F
  7         Plantation Forest       #32CD32
  8        Rubber Monoculture       #8B4513
  9      Oil palm Monoculture       #FF8C00
 10         Cacao Monoculture       #D2691E
 11       Coconut monoculture       #F4A460
 12         Other Monoculture       #DAA520
 13            Other Cropland       #FFFF00
 14       Coffee agroforestry       #6B8E23
 15       Rubber agroforestry       #9ACD32
 16         Mixed/home garden       #7CFC00
 17               Paddy field       #EEE8AA
 18          Grass or Savanna       #ADFF2F
 19                     Shrub       #90EE90
 20                Settlement       #FF0000
 21              Cleared Land       #D2B48C
 22               Mining area   

# 3. Upload modular reference data

In [6]:
import pandas as pd
import numpy as np
import json

# use the modular reference dataset was reverse engineered from KLHK map
df_train_csv = '../data/modular_mapping_approach/sumatra_test/sumatra_extracted_raster_values_by_provinces_CLEANED.csv'
df_train = pd.read_csv(df_train_csv)

# Extract lon and lat
# df_train[['longitude', 'latitude']] = (
#     df_train['geometry']
#     .str.extract(r'POINT\s*\(([-\d.]+)\s+([-\d.]+)\)')
#     .astype(float)
# )

# df_train = df_train.drop(columns=['geometry'])

df_train[['longitude', 'latitude']] = (
    df_train['.geo']
    .apply(lambda x: json.loads(x)['coordinates'])
    .apply(pd.Series)
)

df_train = df_train.drop(columns=['.geo', 'system:index'])

print(df_train.head())

    AoI  ID                    LULC_24  agricultural_activity  \
0  Aceh   1  Hutan lahan kering primer                    0.0   
1  Aceh   1  Hutan lahan kering primer                    0.0   
2  Aceh   1  Hutan lahan kering primer                    0.0   
3  Aceh   1  Hutan lahan kering primer                    0.0   
4  Aceh   1  Hutan lahan kering primer                    0.0   

   artificial_waterbody  bareSoil_cover  builtup_cover  cacao_presence  \
0                   0.0       -0.252114            0.0             0.0   
1                   0.0       -0.361223            0.0             0.0   
2                   0.0       -0.143024            0.0             0.0   
3                   0.0       -0.365143            0.0             0.0   
4                   0.0       -0.267673            0.0             0.0   

   coconut_presence  coffee_presence  ...  oilpalm_presence  paddy_presence  \
0               0.0         0.000000  ...               0.0             0.0   
1     

In [7]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "Epistem"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df.to_string(index=False))

 ID          Land Cover Class Color Palette
  1    Primary Dryland Forest       #006400
  2  Secondary Dryland Forest       #228B22
  3   Primary Mangrove Forest       #4169E1
  4 Secondary Mangrove Forest       #87CEEB
  5      Primary Swamp Forest       #2E8B57
  6    Secondary Swamp Forest       #8FBC8F
  7         Plantation Forest       #32CD32
  8        Rubber Monoculture       #8B4513
  9      Oil palm Monoculture       #FF8C00
 10         Cacao Monoculture       #D2691E
 11       Coconut monoculture       #F4A460
 12         Other Monoculture       #DAA520
 13            Other Cropland       #FFFF00
 14       Coffee agroforestry       #6B8E23
 15       Rubber agroforestry       #9ACD32
 16         Mixed/home garden       #7CFC00
 17               Paddy field       #EEE8AA
 18          Grass or Savanna       #ADFF2F
 19                     Shrub       #90EE90
 20                Settlement       #FF0000
 21              Cleared Land       #D2B48C
 22               Mining area   

## Define default scheme labelling ruleset

In [8]:
# ── DEFINE RULESET ─────────────────────────────────────────────────────────
# Each row = one class rule. Columns are primitives with operators.
# Format: ">0.40" means "greater than 0.40"
#         "==1" means "equal to 1"
#         ">=25" means "greater than or equal to 25"
#         'treecover': '<30 | >80',  means treecover < 30 OR treecover > 80   
#         None means "no condition on this primitive"

ruleset_csv = '../data/modular_mapping_approach/ruleset_epistem_default_v5.csv'
ruleset = pd.read_csv(ruleset_csv)
ruleset = ruleset.sort_values(by='priority', ascending=True)
print(ruleset)

# copy to clipboard for easy pasting into the ruleset CSV file
pd.DataFrame.to_clipboard(ruleset)

CLASS_NAMES = dict(zip(ruleset['class_id'], ruleset['class_name']))
CLASS_NAMES[0] = 'unclassified'
CLASS_NAMES[-1] = 'abstain'

    class_id                 class_name  priority waterbody_cover  \
0         23                  Waterbody         1             ==2   
1         20                  Fish Pond         2             >=1   
2         24                 Settlement         3             NaN   
3         21               Cleared Land         4             NaN   
4         22                Mining area         5             NaN   
5          9       Oil palm monoculture         6             NaN   
6         11        Coconut monoculture         7             NaN   
7         15        Rubber agroforestry         8             NaN   
8          8         Rubber monoculture         9             NaN   
9         14        Coffee agroforestry        10             NaN   
10        10          Cacao monoculture        11             NaN   
11        16          Mixed/home garden        12             NaN   
12         7          Plantation forest        13             NaN   
13        12          Other monocu

## Helper functions to label the classes

In [9]:
def safe_num(val, default=0):
    """Convert value to float, return default if None or NaN."""
    if val is None:
        return default
    if isinstance(val, (int, float)):
        if np.isnan(val):
            return default
        return float(val)
    try:
        return float(val)
    except:
        return default

def evaluate_condition(row_val, condition_str):
    """
    Evaluate a single condition: row_val op threshold?
    Supports OR logic with pipe separator: ">0.40|<0.10"
    
    Args:
        row_val: The value from the sample
        condition_str: String like ">0.40", ">=25", ">0.40|<0.10"
    
    Returns:
        bool: True if condition is satisfied, False otherwise
    """
    if condition_str is None:
        return True  # No condition → always passes
    
    condition_str = str(condition_str).strip()
    
    # Handle OR logic (pipe-separated conditions)
    if '|' in condition_str:
        sub_conditions = [c.strip() for c in condition_str.split('|')]
        return any(evaluate_condition(row_val, c) for c in sub_conditions)

    # Handle AND logic
    if '&' in condition_str:
        sub_conditions = [c.strip() for c in condition_str.split('&')]
        return all(evaluate_condition(row_val, c) for c in sub_conditions)
    
    # Parse operator and threshold
    if condition_str.startswith('=='):
        op, threshold_str = '==', condition_str[2:]
    elif condition_str.startswith('>='):
        op, threshold_str = '>=', condition_str[2:]
    elif condition_str.startswith('<='):
        op, threshold_str = '<=', condition_str[2:]
    elif condition_str.startswith('>'):
        op, threshold_str = '>', condition_str[1:]
    elif condition_str.startswith('<'):
        op, threshold_str = '<', condition_str[1:]
    else:
        return True  # Invalid condition → pass
    
    threshold = safe_num(threshold_str)
    row_val_num = safe_num(row_val)
    
    if op == '>':
        return row_val_num > threshold
    elif op == '>=':
        return row_val_num >= threshold
    elif op == '<':
        return row_val_num < threshold
    elif op == '<=':
        return row_val_num <= threshold
    elif op == '==':
        return np.isclose(
            row_val_num,
            threshold,
            atol=1e-6
        )
    
    return False

def check_rule_match(row, rule):
    """
    Check if a sample row matches all conditions in a rule.
    
    Args:
        row: pd.Series with sample data
        rule: pd.Series with rule conditions
    
    Returns:
        bool: True if ALL conditions are satisfied
    """
    # Get all primitive columns (skip metadata like class_id, class_name, priority)
    metadata = {'class_id', 'class_name', 'priority'}
    primitive_cols = [col for col in rule.index if col not in metadata]
    
    for prim in primitive_cols:
        condition = rule[prim]
        
        # Handle special case for range checks
        if pd.isna(condition) or condition is None:
            continue  # No condition on this primitive
        
        row_val = row.get(prim, np.nan)
        
        if not evaluate_condition(row_val, condition):
            return False  # Any condition fails → rule doesn't match
    
    return True  # All conditions passed

print('✓ Helper functions defined')

✓ Helper functions defined


## Assign the class labels to the modular reference data

Using previous labels to narrow down the potential labels in the new classification scheme

In [10]:
def assign_label(row, ruleset):
    """
    Evaluate rules in priority order.

    - First matching rule returns its class_id.
    - Non-matching rules abstain (-1) and evaluation continues.
    - If every rule abstains, return 0 (unclassified).

    """
    ruleset_sorted = ruleset.sort_values("priority").reset_index(drop=True)

    for _, rule in ruleset_sorted.iterrows():
         if check_rule_match(row, rule):
            return rule["class_id"]  # first vote wins

    return 0  # all LFs abstained

df_train['label'] = df_train.apply(lambda row: assign_label(row, ruleset), axis=1)

# add class_name column
class_mapping = ruleset.set_index('class_id')['class_name'].to_dict()

df_train['class_name'] = df_train['label'].map(class_mapping).fillna('unclassified')

print(f'✓ Labels assigned to {len(df_train)} samples\n')

print('Label distribution (assigned):')
print(
    df_train['class_name']
    .value_counts()
    .reindex(ruleset['class_name'].unique(), fill_value=0)
)

print(f'\nUnclassified (label=0): {(df_train["label"] == 0).sum()}')

✓ Labels assigned to 7078 samples

Label distribution (assigned):
class_name
Waterbody                     289
Fish Pond                     409
Settlement                    107
Cleared Land                   75
Mining area                     1
Oil palm monoculture          750
Coconut monoculture            10
Rubber agroforestry             0
Rubber monoculture             14
Coffee agroforestry             9
Cacao monoculture               7
Mixed/home garden               0
Plantation forest             247
Other monoculture               0
Paddy field                   170
Other Cropland                243
Grass or Savanna             1787
Shrub                           0
Primary Mangrove Forest       314
Primary Swamp Forest            0
Secondary Mangrove Forest     246
Secondary Swamp Forest          0
Primary Dryland Forest       1139
Secondary Dryland Forest     1261
Name: count, dtype: int64

Unclassified (label=0): 0


## Compare with original class label

In [11]:
import numpy as np
import matplotlib.pyplot as plt

ORIGINAL_CLASS = "LULC_24"

print(f"Classified samples: {len(df_train)}")
print()


# Complete list of target classes + unclassified
all_classes = list(ruleset["class_name"].unique()) + ["unclassified"]

# Count how many samples from each original class received each label
summary = (
    pd.crosstab(
        df_train[ORIGINAL_CLASS],
        df_train["class_name"]
    )
    .reindex(columns=all_classes, fill_value=0)
)

print("Original class -> Assigned labels")
print(summary)

summary.to_clipboard(
    excel=True,
    index=True
)

Classified samples: 7078

Original class -> Assigned labels
class_name                              Waterbody  Fish Pond  Settlement  \
LULC_24                                                                    
Bandara/Pelabuhan                               0          0           0   
Hutan lahan kering primer                       0          0           0   
Hutan lahan kering sekunder                     1          0           0   
Hutan mangrove primer                          17          6           0   
Hutan mangrove sekunder/bekas tebangan         29          4           0   
Hutan rawa primer                               2          0           0   
Hutan rawa sekunder/bekas tebangan              0          0           0   
Hutan tanaman                                   1          0           0   
Perkebunan/Kebun                                1          0           4   
Permukiman/lahan terbangun                      2          2          63   
Pertambangan                

## Save the labelled training dataset

In [12]:
import geopandas as gpd
from shapely.geometry import Point

geometry = [Point(xy) for xy in zip(df_train["longitude"], df_train["latitude"])]
gdf = gpd.GeoDataFrame(df_train, geometry=geometry)
gdf.to_file("../data/modular_mapping_approach/sumatra_test/sumatra_td_DTL_result_v5_cleaned.shp")

C:\Users\widijanto\AppData\Local\Temp\ipykernel_4508\2311221451.py:6: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file("../data/modular_mapping_approach/sumatra_test/sumatra_td_DTL_result_v5_cleaned.shp")
c:\Users\widijanto\AppData\Local\miniconda3\envs\luma-ge\Lib\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
c:\Users\widijanto\AppData\Local\miniconda3\envs\luma-ge\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'agricultural_activity' to 'agricultur'
  ogr_write(
c:\Users\widijanto\AppData\Local\miniconda3\envs\luma-ge\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'artificial_waterbody' to 'artificial'
  ogr_write(
c:\Users\widijanto\AppData\Local\miniconda3\envs\luma-ge\Lib\site-packages\pyogrio\ra

## Sanitize relabelled shp file with original Luma module 3 script

In [13]:
from luma_ge.sample_data import SyncTrainData

LULCTable = classification_df
TrainVectPath = "../data/modular_mapping_approach/sumatra_test/sumatra_td_DTL_result.shp"

TrainField = 'labels' 
        # Load and process training data
TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=LULCTable,
            aoi_geometry=aoi,
            training_shp_path=TrainVectPath
        )

2026-08-07 11:25:45,134 - luma_ge.sample_data - INFO - Loading training data from shapefile: ../data/modular_mapping_approach/sumatra_test/sumatra_td_DTL_result.shp
2026-08-07 11:25:45,260 - luma_ge.sample_data - WARNING - 'kelas' field not found in training data
2026-08-07 11:25:45,262 - luma_ge.sample_data - INFO - Available columns: ['ID', 'LULC_24', 'agricultur', 'bareSoil_c', 'builtup_co', 'cacao_pres', 'coffee_pre', 'herb_cover', 'mangrove_p', 'mining', 'oilpalm_pr', 'paddy_pres', 'shrub_cove', 'timber_ext', 'tree_cover', 'tree_heigh', 'waterbody_', 'longitude', 'latitude', 'label', 'class_name', 'geometry']


In [14]:
# ----- System response 3.2.a -----
# Set class field
TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)

# Validate classes
TrainDataDict = SyncTrainData.ValidClass(TrainDataDict,1)

    # Check sample sufficiency
TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)

    # Filter by AOI
TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

    # Create training data table
table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
    training_data=TrainDataDict.get('training_data'),
    landcover_df=TrainDataDict.get('landcover_df'),
    class_field=TrainDataDict.get('class_field'))

#Summary result
vr = TrainDataDict.get('validation_results', {})

print("=" * 70)
print("TRAINING DATA SUMMARY")
print("=" * 70)
print(f"Total training points loaded     : {vr.get('total_points', 'N/A')}")
print(f"Points after class filtering     : {vr.get('points_after_class_filter', 'N/A')}")
print(f"Valid points (inside AOI)        : {vr.get('valid_points', 'N/A')}")
print(f"Invalid classes found            : {len(vr.get('invalid_classes', []))}")
print(f"Points outside AOI               : {len(vr.get('outside_aoi', []))}")
print("=" * 70)

    # --- Display the main table ---
if table_df is not None and not table_df.empty:
        display_df = table_df.copy()
        if 'Percentage' in display_df.columns:
            display_df['Percentage'] = display_df['Percentage'].apply(
                lambda x: f"{x:.2f}%" if isinstance(x, (int, float)) else x
            )
        display(display_df)
else:
        print("No valid training data available to display.")

TrainDataFinal = TrainDataDict.get('training_data')

2026-08-07 11:25:45,284 - luma_ge.sample_data - INFO - Validating classes with use_class_ids=1
2026-08-07 11:25:45,286 - luma_ge.sample_data - INFO - Class field: labels
2026-08-07 11:25:45,288 - luma_ge.sample_data - INFO - Training data type: <class 'geopandas.geodataframe.GeoDataFrame'>
2026-08-07 11:25:45,291 - luma_ge.sample_data - INFO - Landcover DF columns: ['ID', 'Land Cover Class', 'Color Palette']
2026-08-07 11:25:45,293 - luma_ge.sample_data - INFO - Valid IDs in landcover_df: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
2026-08-07 11:25:45,295 - luma_ge.sample_data - INFO - Processing GeoDataFrame with 7098 features
2026-08-07 11:25:45,296 - luma_ge.sample_data - ERROR - Class field 'labels' not found in training data columns: ['ID', 'LULC_24', 'agricultur', 'bareSoil_c', 'builtup_co', 'cacao_pres', 'coffee_pre', 'herb_cover', 'mangrove_p', 'mining', 'oilpalm_pr', 'paddy_pres', 'shrub_cove', 'timber_ext', 'tree_cover', 'tree_heigh

TRAINING DATA SUMMARY
Total training points loaded     : 7098
Points after class filtering     : 7098
Valid points (inside AOI)        : 7098
Invalid classes found            : 0
Points outside AOI               : 0
No valid training data available to display.


# 6. Land cover classification

## Generate multiprobability classification map

In [15]:
from ee import classifier

from luma_ge.classification import FeatureExtraction

labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

#Perform Training Test Split
features = FeatureExtraction()
strafied_train, stratified_test = features.stratified_split(labeled_roi, stacked_landsat, 
                            class_prop='ID', train_ratio=1)

# create classifier in multiprobability output mode
clf_sumatra_prob = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    minLeafPopulation=1
).setOutputMode('MULTIPROBABILITY').train(
        features=strafied_train,
        classProperty='ID',
        inputProperties=stacked_landsat.bandNames()
        )

2026-08-07 11:25:45,557 - pyogrio._io - INFO - Created 7,098 records


Stratified Random Split Training Pixel Size: 7098
Stratified Random Split Testing Pixel Size: 0


In [16]:
# rename band names of the probability
import re


def sanitize_band_name(name: str) -> str:
    """GEE band names should avoid spaces/special chars for safety downstream."""
    name = str(name).strip()
    name = re.sub(r'[^\w]+', '_', name)   # replace non-word chars with underscore
    name = re.sub(r'_+', '_', name).strip('_')
    return name

classification_df = classification_df.sort_values("ID").reset_index(drop=True)
class_labels = [sanitize_band_name(name) for name in classification_df["Land Cover Class"]]

#classify probability
probability_stack = stacked_landsat.select(stacked_landsat.bandNames()).classify(clf_sumatra_prob)
probability_stack = probability_stack.arrayFlatten([class_labels])

# # Check the final GEE band names
# print("\nFinal probability stack band names:")
# print(probability_stack.bandNames().getInfo())

# 7. Export Multiprobability stack

Create loop per provinces because exporting them all at once per island keeps failing

In [17]:
# Province boundaries
provinces = ee.FeatureCollection('projects/epistem2/assets/AOI_Sumatra_Provinces')

# Get province names and geometries
province_list = provinces.toList(provinces.size())

n_provinces = provinces.size().getInfo()

for i in range(n_provinces):

    province = ee.Feature(province_list.get(i))

    # Change 'AoI' to the actual province-name field
    province_name = province.get('AoI').getInfo()

    province_geom = province.geometry()

    # Clean province name for use in task/file names
    province_name_clean = (
        province_name
        .replace(' ', '_')
        .replace('/', '_')
        .replace('-', '_')
    )

    print(f"Creating export for: {province_name}")

    task = ee.batch.Export.image.toDrive(
        image=probability_stack.clip(province_geom),
        description=f'sumatra_probability_{province_name_clean}_Epistem_v5',
        folder='sumatra_multiprobability_stack_Epistem',
        fileNamePrefix=f'sumatra_probability_{province_name_clean}_Epistem_v5',
        region=province_geom,
        scale=100,
        crs='EPSG:4326',
        maxPixels=1e13,
        shardSize=8,
        fileFormat='GeoTIFF',
        formatOptions={
            'cloudOptimized': True
        }
    )

    task.start()

    print(f"  ✓ Started: {province_name_clean}")

Creating export for: Aceh
  ✓ Started: Aceh
Creating export for: Bengkulu
  ✓ Started: Bengkulu
Creating export for: Jambi
  ✓ Started: Jambi
Creating export for: Kepulauan Bangka Belitung
  ✓ Started: Kepulauan_Bangka_Belitung
Creating export for: Kepulauan Riau
  ✓ Started: Kepulauan_Riau
Creating export for: Lampung
  ✓ Started: Lampung
Creating export for: Riau
  ✓ Started: Riau
Creating export for: Sumatera Barat
  ✓ Started: Sumatera_Barat
Creating export for: Sumatera Selatan
  ✓ Started: Sumatera_Selatan
Creating export for: Sumatera Utara
  ✓ Started: Sumatera_Utara
